# House Price Prediction

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_log_error, r2_score

def compute_rmsle(y_test, y_pred, precision=2):
    rmsle = np.sqrt(mean_squared_log_error(y_test, y_pred))
    return round(rmsle, precision)

print('Setup Complete')

Setup Complete


## Load Data

In [2]:
train_data = pd.read_csv('../data/train.csv')
test_data = pd.read_csv('../data/test.csv')
print('Data loaded')

Data loaded


## Feature Selection

In [3]:
continuous_features = ['LotArea', 'YearBuilt', 'OverallQual', 'OverallCond', 'GrLivArea', 'TotalBsmtSF', 'GarageArea', '1stFlrSF']
categorical_features = ['MSZoning', 'Neighborhood', 'BldgType', 'HouseStyle', 'SaleCondition', 'SaleType']
continuous_features = [col for col in continuous_features if col in train_data.columns]
categorical_features = [col for col in categorical_features if col in train_data.columns]
print(f'Features Selected - {len(continuous_features)} continuous, {len(categorical_features)} categorical')

Features Selected - 8 continuous, 6 categorical


## Data Preprocessing

In [4]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

all_features = continuous_features + categorical_features
train_clean = train_data.dropna(subset=['SalePrice'])
X = train_clean[all_features].copy()
y = train_clean['SalePrice'].copy()

for col in continuous_features:
    if X[col].isnull().sum() > 0:
        X[col].fillna(X[col].mean(), inplace=True)

for col in categorical_features:
    if X[col].isnull().sum() > 0:
        X[col].fillna(X[col].mode()[0], inplace=True)

X_encoded = X.copy()
for col in categorical_features:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))

scaler = StandardScaler()
X_encoded[continuous_features] = scaler.fit_transform(X_encoded[continuous_features])

print('Data preprocessed and encoded')

Data preprocessed and encoded


## Model Training

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

print(f'Model trained on {X_train.shape[0]} samples')

Model trained on 1168 samples


## Model Evaluation

In [6]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

y_train_pred = np.maximum(y_train_pred, 1)
y_test_pred = np.maximum(y_test_pred, 1)

train_rmsle = compute_rmsle(y_train, y_train_pred)
test_rmsle = compute_rmsle(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print('='*50)
print('MODEL EVALUATION METRICS')
print('='*50)
print('TRAINING SET:')
print(f'  RMSLE:  {train_rmsle:.2f}  (Primary metric)')
print(f'  R2:     {train_r2:.2f}')
print('TEST SET:')
print(f'  RMSLE:  {test_rmsle:.2f}  (Primary metric)')
print(f'  R2:     {test_r2:.2f}')

MODEL EVALUATION METRICS
TRAINING SET:
  RMSLE:  0.55  (Primary metric)
  R2:     0.88
TEST SET:
  RMSLE:  0.85  (Primary metric)
  R2:     0.82


## Visualizations

In [7]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_train, y_train_pred, alpha=0.5)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0].set_title(f'Training Set (RMSLE={train_rmsle:.4f})')
axes[1].scatter(y_test, y_test_pred, alpha=0.5, color='green')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_title(f'Test Set (RMSLE={test_rmsle:.4f})')
plt.show()

print('Visualizations created')

Visualizations created


## Conclusions

- Built Linear Regression model
- Used 8 continuous and 6 categorical features
- Proper scaling and encoding
- RMSLE metric for evaluation